# Minibatch Training: From Basic Loops to PyTorch DataLoaders

## What is this notebook about?

This notebook teaches you how to **train a neural network properly**. While previous notebooks covered forward passes and backpropagation, this notebook covers everything else needed for real training:

1. **Cross-Entropy Loss**: The correct loss function for classification problems
2. **Training Loops**: How to iterate through data and update weights
3. **Optimizers**: Using PyTorch's built-in optimization algorithms
4. **Datasets & DataLoaders**: Efficiently loading and batching data
5. **Validation**: Measuring how well your model generalizes to new data

---

## Key Concepts (Glossary)

Before we dive in, let's define some important terms you'll encounter throughout this notebook:

| Term | Definition |
|------|------------|
| **Minibatch** | A small subset of training data processed together (e.g., 50 images at a time) |
| **Epoch** | One complete pass through all training data |
| **Cross-Entropy** | Loss function specifically designed for classification problems |
| **Softmax** | Function that converts raw scores to probabilities that sum to 1 |
| **Logits** | Raw output scores from the model before converting to probabilities |
| **Learning Rate** | How big a step we take when updating weights (a hyperparameter) |
| **Optimizer** | Algorithm that updates weights (SGD, Adam, etc.) |
| **Dataset** | Abstraction for accessing data samples |
| **DataLoader** | Batches and shuffles data for training |
| **Gradient** | The direction and magnitude of change needed for each weight |

---

## The Training Loop Overview

Here's the big picture of what happens during training:

```
+-----------------------------------------------------------------------------+
|                         THE TRAINING LOOP                                   |
+-----------------------------------------------------------------------------+
|                                                                             |
|   FOR each epoch (complete pass through data):                              |
|   |                                                                         |
|   +-- FOR each minibatch (small chunk of data):                             |
|   |   |                                                                     |
|   |   +-- 1. FORWARD PASS:   predictions = model(inputs)                   |
|   |   |                      Run data through the network                   |
|   |   |                                                                     |
|   |   +-- 2. COMPUTE LOSS:   loss = loss_func(predictions, targets)        |
|   |   |                      Measure how wrong we are                       |
|   |   |                                                                     |
|   |   +-- 3. BACKWARD PASS:  loss.backward()                               |
|   |   |                      Compute gradients for all parameters          |
|   |   |                                                                     |
|   |   +-- 4. UPDATE WEIGHTS: optimizer.step()                              |
|   |   |                      Adjust weights to reduce loss                  |
|   |   |                                                                     |
|   |   +-- 5. ZERO GRADIENTS: optimizer.zero_grad()                         |
|   |                          Clear gradients for next iteration             |
|   |                                                                         |
|   +-- VALIDATE: Check performance on held-out data                          |
|                                                                             |
+-----------------------------------------------------------------------------+
```

---

## Why Use Minibatches?

You might wonder: why not just train on all data at once? Here's why minibatches are essential:

- **Memory Constraints**: Modern datasets are huge! You can't fit 50,000 images in GPU memory at once
- **Faster Updates**: With minibatches, we update weights more frequently (1000 updates vs 1 per epoch)
- **Better Generalization**: The "noise" from minibatch gradients helps escape poor local minima
- **Parallelization**: GPUs are optimized for parallel computation on batches of data

Typical batch sizes range from 16 to 256 depending on your GPU memory and task.

## Google Colab Setup

Run the cells below **once** at the start of each Colab session. They mount Google Drive, set the working directory, install the standard packages, and clone the `miniai` library from the fast.ai Part 2 course repo so the `from miniai.X import *` lines work.

**GPU note:** All examples run fine on CPU; no GPU needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Fast.AI_Colab')
print(os.getcwd())

In [ ]:
# Install required packages
!pip install -q fastcore fastai diffusers datasets torcheval accelerate

# Clone the fast.ai Part 2 course repo to get the miniai library
if not os.path.exists('course22p2'):
    !git clone https://github.com/fastai/course22p2.git

# Add miniai to the Python path
import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'course22p2'))

# Verify miniai is accessible
try:
    import miniai
    print(f'miniai loaded successfully from: {miniai.__file__}')
except ImportError:
    print('ERROR: miniai not found. Check that course22p2 was cloned correctly.')

In [ ]:
!pip install nbdev

---

*The cells above are Colab-specific setup. The content below is the same as the source notebook (`04_minibatch_training_explained.ipynb`), unchanged.*

---

In [ ]:
# ============================================================================
# NBDEV EXPORT DIRECTIVE
# ============================================================================
# This special comment tells nbdev to export code from this notebook
# to a Python module. This is how the fast.ai library is built!
#
# What is nbdev?
# - nbdev is a tool that lets you write Python libraries in Jupyter notebooks
# - Code cells marked with #|export get extracted to .py files
# - This notebook exports to a file called "training.py"
#
# You don't need to worry about this for learning - it's just for library development.

#| default_exp training

In [ ]:
# ============================================================================
# IMPORTING REQUIRED LIBRARIES
# ============================================================================
# The #|export comment marks this code for export to the training module

#|export
import pickle      # For loading serialized Python objects (our dataset is saved as a pickle file)
import gzip        # For reading compressed .gz files (our dataset is gzip compressed)
import math        # Mathematical functions (we'll use this for some calculations)
import os          # Operating system functions (file paths, etc.)
import time        # For timing code execution
import shutil      # File operations (copy, move, delete)

import torch                    # PyTorch - our deep learning framework
import matplotlib as mpl        # Plotting configuration
import numpy as np              # Numerical operations (we'll use PyTorch mostly)
import matplotlib.pyplot as plt # Plotting functions for visualizations

from pathlib import Path        # Modern file path handling (cleaner than os.path)
from torch import tensor, nn   # tensor: creates tensors, nn: neural network module
import torch.nn.functional as F # Functional operations (loss functions, activations)

# ============================================================================
# UNDERSTANDING THE IMPORTS
# ============================================================================
#
# torch.nn (imported as nn):
#   - Contains neural network building blocks (Linear, Conv2d, ReLU, etc.)
#   - Base classes for creating custom layers (nn.Module)
#   - Built-in loss functions and optimizers
#   - Example: nn.Linear(10, 5) creates a layer that transforms 10 inputs to 5 outputs
#
# torch.nn.functional (imported as F):
#   - Contains the SAME operations as nn, but in "functional" form
#   - F.relu(x) vs nn.ReLU()(x) - same result, different style
#   - F.cross_entropy(), F.softmax(), F.log_softmax(), etc.
#   - Use functional when you don't need to store state (no learnable parameters)
#
# When to use nn vs F:
#   - Use nn.Module when the layer has learnable parameters (weights, biases)
#   - Use F when you just need a stateless transformation
#   - Example: nn.Linear for weights, F.relu for activation

In [ ]:
# ============================================================================
# SETUP AND LOAD DATA
# ============================================================================

from fastcore.test import test_close  # Utility to test if two values are close enough

# ============================================================================
# DISPLAY SETTINGS
# ============================================================================
# These settings make tensor output easier to read in the notebook

# precision=2: Show only 2 decimal places (instead of many)
# Example: 0.123456789 becomes 0.12
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
#   linewidth=140: Allow 140 characters per line when printing tensors
#   sci_mode=False: Don't use scientific notation (show 0.0001 instead of 1e-4)

# Set random seed for reproducibility
# When you set a seed, "random" numbers become predictable
# Same seed = same "random" sequence = reproducible results
# This is crucial for debugging and comparing experiments!
torch.manual_seed(1)

# Display images in grayscale (MNIST digits are black and white)
mpl.rcParams['image.cmap'] = 'gray'

# ============================================================================
# LOAD MNIST DATASET
# ============================================================================
# MNIST is a famous dataset of handwritten digits (0-9)
# It contains:
#   - 60,000 training images (we'll use 50,000 for training, 10,000 for validation)
#   - 10,000 test images
#   - Each image is 28x28 pixels = 784 values
#   - Each pixel is a grayscale value from 0 (black) to 1 (white)
#   - Labels are integers 0-9 representing the digit

path_data = Path('data')          # Directory where data is stored
path_gz = path_data/'mnist.pkl.gz' # Path to the compressed pickle file

# Load the pickled data from compressed file
# pickle.load() deserializes (unpacks) the Python objects saved in the file
# The file was created by someone who saved: ((train_x, train_y), (valid_x, valid_y), _)
with gzip.open(path_gz, 'rb') as f:  # 'rb' = read binary mode
    ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding='latin-1')

# Convert NumPy arrays to PyTorch tensors
# PyTorch requires tensors for all operations
# map() applies the tensor() function to each array in the list
x_train, y_train, x_valid, y_valid = map(tensor, [x_train, y_train, x_valid, y_valid])

print("Data loaded successfully!")
print(f"Training set:   {x_train.shape[0]:,} images, each with {x_train.shape[1]} pixels")
print(f"Validation set: {x_valid.shape[0]:,} images")
print(f"\nFirst training image shape: {x_train[0].shape}")
print(f"First training label: {y_train[0]} (this image shows the digit '{y_train[0].item()}')")

---

## Part 1: Understanding Our Data and Building a Simple Model

Before we can train a neural network, we need to:
1. Understand the dimensions of our data
2. Build a simple model architecture
3. Define a proper loss function for classification

Let's start by exploring our data dimensions.

In [ ]:
# ============================================================================
# UNDERSTANDING DATA DIMENSIONS
# ============================================================================
# These are the key dimensions we need to know for building our model

n, m = x_train.shape  # Unpack the 2D shape into two variables
# n = number of training samples (how many images we have)
# m = number of features per sample (how many pixels per image)

print(f"x_train.shape = {x_train.shape}")
print(f"  n = {n:,} (number of training images)")
print(f"  m = {m} (pixels per image = 28 x 28)")

# Number of output classes
# Our labels are 0, 1, 2, ..., 9 (ten different digits)
# The maximum label is 9, so we have 9 + 1 = 10 classes
c = y_train.max() + 1
print(f"\nc = {c} (number of classes: digits 0-9)")

# Number of hidden neurons (this is a HYPERPARAMETER)
# A hyperparameter is a value WE choose, not something the model learns
# More neurons = more capacity to learn complex patterns
# Fewer neurons = faster training, less risk of overfitting
# 50 is a reasonable starting point for this simple problem
nh = 50
print(f"nh = {nh} (hidden neurons - we chose this)")

print("\n" + "="*60)
print("SUMMARY: Our model will be:")
print(f"  Input:  {m} neurons (one per pixel)")
print(f"  Hidden: {nh} neurons (we chose this)")
print(f"  Output: {c} neurons (one per digit class)")
print("="*60)

### Let's Visualize Some Training Data

Before building our model, let's look at what our data actually looks like. Remember, each image is stored as a flat vector of 784 numbers, but originally it's a 28x28 pixel image.

In [ ]:
# ============================================================================
# VISUALIZING SAMPLE IMAGES
# ============================================================================

# Create a figure with 5 sample images side by side
fig, axes = plt.subplots(1, 5, figsize=(12, 3))

for i, ax in enumerate(axes):
    # Get one image and reshape from flat (784,) to 2D (28, 28)
    # view() reshapes the tensor - same data, different shape
    img = x_train[i].view(28, 28)

    # Display the image
    ax.imshow(img)
    ax.set_title(f"Label: {y_train[i].item()}")
    ax.axis('off')  # Hide axis ticks and numbers

plt.suptitle("Sample Training Images from MNIST", fontsize=14)
plt.tight_layout()
plt.show()

print("Each image is 28x28 = 784 pixels")
print("Pixel values range from 0 (black) to ~1 (white)")
print("Our model sees these as flat vectors of 784 numbers")

### Building Our Neural Network Model

Now let's build a simple 2-layer neural network. This is similar to what we built in previous notebooks, but now the output layer has 10 neurons (one for each digit class) instead of 1.

**Architecture:**
```
Input (784 pixels) --> Linear --> ReLU --> Linear --> Output (10 classes)
```

**Key Concept - Logits:**
The output of our model is called "logits" - these are raw scores, NOT probabilities. A larger logit means the model is more confident about that class. We'll convert these to probabilities later using the softmax function.

In [ ]:
# ============================================================================
# SIMPLE NEURAL NETWORK MODEL
# ============================================================================

class Model(nn.Module):
    """
    A simple 2-layer neural network for digit classification.

    Architecture:
        Input (784) --> Linear --> ReLU --> Linear --> Output (10)

    The output layer has 10 neurons - one for each digit (0-9).
    These outputs are called "logits" - raw scores before softmax.
    """

    def __init__(self, n_in, nh, n_out):
        """
        Initialize the model layers.

        Parameters:
        -----------
        n_in : int
            Number of input features (784 for MNIST - one per pixel)
        nh : int
            Number of hidden neurons (50 in our case)
        n_out : int
            Number of output classes (10 for digits 0-9)
        """
        # Call the parent class (nn.Module) constructor
        # This is REQUIRED for PyTorch to track parameters properly
        super().__init__()

        # Create the layers as a list
        # We'll pass data through each layer in sequence
        self.layers = [
            nn.Linear(n_in, nh),  # First layer: 784 inputs --> 50 hidden neurons
            nn.ReLU(),            # Activation function (introduces non-linearity)
            nn.Linear(nh, n_out)  # Second layer: 50 hidden --> 10 output classes
        ]

        # ================================================================
        # What each layer does:
        # ================================================================
        # nn.Linear(784, 50):
        #   - Has 784 * 50 = 39,200 weights + 50 biases = 39,250 parameters
        #   - Computes: output = input @ weights.T + bias
        #   - Transforms 784-dimensional input to 50-dimensional output
        #
        # nn.ReLU():
        #   - ReLU(x) = max(0, x)
        #   - Keeps positive values, sets negative values to 0
        #   - This non-linearity is what makes neural networks powerful!
        #   - Without it, stacking linear layers would collapse to one linear layer
        #
        # nn.Linear(50, 10):
        #   - Has 50 * 10 = 500 weights + 10 biases = 510 parameters
        #   - Outputs 10 values (logits), one per digit class

    def __call__(self, x):
        """
        Forward pass: compute predictions for input x.

        Parameters:
        -----------
        x : tensor
            Input batch of images, shape (batch_size, 784)

        Returns:
        --------
        tensor
            Logits (raw scores), shape (batch_size, 10)
            Each row has 10 values - one score per digit class
        """
        # Pass through each layer in sequence
        for layer in self.layers:
            x = layer(x)  # Output of one layer becomes input to the next
        return x  # Return logits (NOT probabilities yet!)

# Create model instance
model = Model(m, nh, 10)

print("Model created!")
print(f"Architecture: {m} --> {nh} --> 10")
print(f"\nLayers:")
for i, layer in enumerate(model.layers):
    print(f"  {i}: {layer}")

# Count parameters
total_params = sum(p.numel() for layer in model.layers if hasattr(layer, 'weight')
                   for p in [layer.weight, layer.bias])
print(f"\nTotal trainable parameters: {total_params:,}")

In [ ]:
# ============================================================================
# TEST THE MODEL: FORWARD PASS
# ============================================================================
# Let's run all training data through the model to see what it outputs.
# This is called a "forward pass" - data flows forward through the network.

# Run ALL 50,000 training images through the model
# Note: In practice, we'd use minibatches, but this is just a test
pred = model(x_train)

print("Forward pass complete!")
print(f"Input shape:  {x_train.shape}  (50000 images, 784 pixels each)")
print(f"Output shape: {pred.shape}     (50000 images, 10 logits each)")

print("\n" + "="*60)
print("UNDERSTANDING THE OUTPUT:")
print("="*60)
print("Each image now has 10 output values (one per digit class 0-9)")
print("These are called LOGITS - raw scores, NOT probabilities!")
print("\nExample - First image's logits:")
print(pred[0])
print(f"\nTrue label for first image: {y_train[0].item()}")
print("\nThe model hasn't been trained yet, so these are essentially random.")
print("After training, the logit for the correct class should be highest.")

---

## Part 2: Cross-Entropy Loss for Classification

For classification problems, we use **cross-entropy loss** instead of MSE (Mean Squared Error). Cross-entropy is specifically designed for picking one class out of many!

### Step 1: Softmax - Converting Logits to Probabilities

The **softmax** function converts raw scores (logits) into probabilities that:
1. Are all positive (between 0 and 1)
2. Sum to 1.0 (like proper probabilities should)
3. Preserve the ordering (larger logits become larger probabilities)

**The Softmax Formula:**

$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

**Step-by-step Example:**

If logits are `[2.0, 1.0, 0.1]` (for a 3-class problem):

1. **Compute exponentials:** `exp([2.0, 1.0, 0.1]) = [7.39, 2.72, 1.10]`
2. **Sum them:** `7.39 + 2.72 + 1.10 = 11.21`
3. **Divide each by sum:** `[7.39/11.21, 2.72/11.21, 1.10/11.21] = [0.66, 0.24, 0.10]`

Result: **66% confident it's class 0, 24% class 1, 10% class 2** (sums to 100%)

```
+-----------------------------------------------------------------------------+
|                    FROM LOGITS TO PROBABILITIES                              |
+-----------------------------------------------------------------------------+
|                                                                             |
|   Logits (raw scores):    [2.0,  1.0,  0.1]   <-- Can be any real numbers  |
|                              |                                              |
|                              v                                              |
|   Exponentials:           [7.39, 2.72, 1.10]  <-- All positive now         |
|                              |                                              |
|                              v                                              |
|   Divide by sum (11.21):  [0.66, 0.24, 0.10]  <-- Sum to 1.0!             |
|                                                                             |
|   Interpretation: 66% class 0, 24% class 1, 10% class 2                    |
|                                                                             |
+-----------------------------------------------------------------------------+
```

In [ ]:
# ============================================================================
# UNDERSTANDING SOFTMAX WITH A SIMPLE EXAMPLE
# ============================================================================

# Let's work through a simple example step by step
example_logits = torch.tensor([2.0, 1.0, 0.1])

print("="*60)
print("SOFTMAX STEP-BY-STEP EXAMPLE")
print("="*60)
print(f"Starting with logits: {example_logits.tolist()}")

# Step 1: Compute exponentials
# exp() is the exponential function: e^x where e ≈ 2.718
# This makes all values positive and emphasizes differences
exponentials = example_logits.exp()
print(f"\nStep 1 - Compute exp() of each value:")
print(f"  exp(2.0) = e^2.0 = {torch.exp(torch.tensor(2.0)):.2f}")
print(f"  exp(1.0) = e^1.0 = {torch.exp(torch.tensor(1.0)):.2f}")
print(f"  exp(0.1) = e^0.1 = {torch.exp(torch.tensor(0.1)):.2f}")
print(f"  Result: {exponentials.tolist()}")

# Step 2: Sum them up
exp_sum = exponentials.sum()
print(f"\nStep 2 - Sum all exponentials:")
print(f"  7.39 + 2.72 + 1.11 = {exp_sum:.2f}")

# Step 3: Divide each by the sum
probabilities = exponentials / exp_sum
print(f"\nStep 3 - Divide each by the sum to get probabilities:")
print(f"  7.39 / 11.21 = {probabilities[0]:.2f}")
print(f"  2.72 / 11.21 = {probabilities[1]:.2f}")
print(f"  1.11 / 11.21 = {probabilities[2]:.2f}")
print(f"  Result: {[round(p.item(), 2) for p in probabilities]}")

# Verify they sum to 1
print(f"\nVerification: Sum of probabilities = {probabilities.sum():.4f}")

print("\n" + "="*60)
print("KEY INSIGHTS ABOUT SOFTMAX:")
print("="*60)
print("1. Transforms ANY values into valid probabilities")
print("2. Larger logits -> Larger probabilities")
print("3. Output always sums to 1.0")
print("4. Uses exp() to make all values positive first")

### Why Use Log-Softmax Instead of Softmax?

In practice, we almost always compute **log(softmax)** instead of just softmax. Here's why:

**1. Numerical Stability:**
- Softmax can produce very small numbers (close to 0)
- When we take log later for the loss, log(tiny number) = very negative number
- This can cause numerical underflow (numbers too small to represent)

**2. Computational Efficiency:**
- The loss function needs log(probabilities)
- By computing log(softmax) directly, we combine two operations into one
- This is faster and more accurate

**3. Mathematical Simplification:**
- log(softmax) has a simpler formula than computing softmax then taking log

In [ ]:
# ============================================================================
# LOG-SOFTMAX: NAIVE IMPLEMENTATION
# ============================================================================
# This is the direct formula: log(softmax(x)) = log(exp(x) / sum(exp(x)))

def log_softmax_naive(x):
    """
    Compute log-softmax (naive version).

    Parameters:
    -----------
    x : tensor
        Input tensor of logits, shape (batch_size, num_classes)

    Returns:
    --------
    tensor
        Log-probabilities, shape (batch_size, num_classes)
        All values will be negative or zero (since log(p) <= 0 when p <= 1)

    Step by step:
        1. x.exp()                           -> Compute exp(x) for each element
        2. x.exp().sum(-1, keepdim=True)     -> Sum along last dimension (classes)
                                                keepdim=True keeps shape for broadcasting
        3. exp(x) / sum(exp(x))              -> Softmax probabilities
        4. .log()                            -> Take log of probabilities
    """
    # Compute softmax first
    softmax_probs = x.exp() / x.exp().sum(-1, keepdim=True)
    # Then take log
    return softmax_probs.log()

# Test on a small example
# 2 samples, 3 classes each
example = torch.tensor([[2.0, 1.0, 0.1],
                        [1.0, 2.0, 3.0]])

result = log_softmax_naive(example)

print("Input logits (2 samples, 3 classes):")
print(example)
print("\nLog-softmax output:")
print(result)
print("\nNotice: All values are NEGATIVE")
print("This is because log(probability) is always <= 0 since probability <= 1")
print("  - log(1.0) = 0 (max possible)")
print("  - log(0.5) = -0.69")
print("  - log(0.1) = -2.30")
print("\nConverting back to probabilities (exp of log-softmax):")
print(result.exp())
print(f"Sum of each row (should be 1.0): {result.exp().sum(dim=1)}")

### Simplifying Log-Softmax Using Log Properties

We can simplify the log-softmax formula using the property:

$$\log(a/b) = \log(a) - \log(b)$$

Starting from:
$$\log\left(\frac{e^{x_i}}{\sum_j e^{x_j}}\right)$$

We get:
$$= \log(e^{x_i}) - \log\left(\sum_j e^{x_j}\right)$$
$$= x_i - \log\left(\sum_j e^{x_j}\right)$$

This is more efficient because:
- We only compute `exp()` once (not twice like the naive version)
- We use subtraction instead of division (faster on computers)

In [ ]:
# ============================================================================
# LOG-SOFTMAX: SIMPLIFIED VERSION
# ============================================================================
# Using log(a/b) = log(a) - log(b), we get a simpler, more efficient formula
# log(exp(x) / sum(exp(x))) = log(exp(x)) - log(sum(exp(x)))
#                           = x - log(sum(exp(x)))

def log_softmax_v2(x):
    """
    Compute log-softmax (simplified version).

    Formula: log_softmax(x)_i = x_i - log(sum(exp(x)))

    This is more efficient because:
    1. We only compute exp() once (not twice like the naive version)
    2. Subtraction is computationally cheaper than division
    """
    # x.exp() - compute exp for each element
    # .sum(-1, keepdim=True) - sum along last dimension, keep shape for broadcasting
    # .log() - take log of the sum
    # x - ... - subtract from original logits
    return x - x.exp().sum(-1, keepdim=True).log()

# Verify it gives the same result as the naive version
example = torch.tensor([[2.0, 1.0, 0.1],
                        [1.0, 2.0, 3.0]])

naive_result = log_softmax_naive(example)
simplified_result = log_softmax_v2(example)

print("Comparing naive vs simplified log_softmax:")
print("\nNaive version:")
print(naive_result)
print("\nSimplified version:")
print(simplified_result)
print(f"\nAre they equal? {torch.allclose(naive_result, simplified_result)}")
print("\nThe simplified version is more efficient - same result, fewer operations!")

### The Problem with Large Logits: Numerical Overflow

There's still a problem with our log-softmax: if logits are very large (e.g., 1000), then `exp(1000)` overflows to infinity!

Let's see this problem in action:

In [ ]:
# ============================================================================
# DEMONSTRATING THE OVERFLOW PROBLEM
# ============================================================================

# What happens with large logits?
large_logits = torch.tensor([1000.0, 1001.0, 1002.0])

print("Large logits:", large_logits.tolist())
print("\nTrying to compute exp():")
print(f"  exp(1000) = {torch.exp(torch.tensor(1000.0))}")  # inf!
print("  Result is 'inf' (infinity) - the number is too large!")

# This causes problems in log_softmax
result = log_softmax_v2(large_logits.unsqueeze(0))  # unsqueeze adds batch dimension
print(f"\nlog_softmax result: {result}")

print("\nThe result contains 'nan' (not a number)!")
print("This happens because inf/inf is undefined.")
print("\nWe need a more numerically stable approach...")

### The LogSumExp Trick: Numerical Stability

The **LogSumExp trick** solves the overflow problem by subtracting the maximum value first:

$$\log\left(\sum_j e^{x_j}\right) = \max(x) + \log\left(\sum_j e^{x_j - \max(x)}\right)$$

**Why this works:**
1. After subtracting max, all exponents become <= 0
2. exp(0) = 1, exp(negative) < 1
3. No overflow possible because exp(negative) is always a small positive number!

**Mathematical proof:**
$$\log\sum e^{x_j} = \log\left(e^{\max(x)} \cdot \sum e^{x_j - \max(x)}\right)$$
$$= \log(e^{\max(x)}) + \log\sum e^{x_j - \max(x)}$$
$$= \max(x) + \log\sum e^{x_j - \max(x)}$$

In [ ]:
# ============================================================================
# LOGSUMEXP: NUMERICALLY STABLE VERSION
# ============================================================================

def logsumexp(x):
    """
    Compute log(sum(exp(x))) in a numerically stable way.

    The trick: log(sum(exp(x))) = max(x) + log(sum(exp(x - max(x))))

    By subtracting the max first, we prevent overflow because:
    - All values become <= 0 after subtracting max
    - exp(0) = 1, exp(negative) < 1
    - No overflow possible!

    Parameters:
    -----------
    x : tensor
        Input tensor, shape (batch_size, num_classes)

    Returns:
    --------
    tensor
        LogSumExp values, shape (batch_size,)
    """
    # Get the maximum value along the last dimension (the classes)
    # max() returns (values, indices) tuple - we only need values [0]
    m = x.max(-1)[0]

    # Subtract max, compute exp, sum, take log, add max back
    # m[:, None] reshapes m from (batch_size,) to (batch_size, 1) for broadcasting
    return m + (x - m[:, None]).exp().sum(-1).log()

# Test with large logits - now it works!
large_logits = torch.tensor([[1000.0, 1001.0, 1002.0]])

print("Large logits:", large_logits.tolist())
print("\nLet's trace through the LogSumExp trick step by step:")

# Step by step
m = large_logits.max(-1)[0]
print(f"\nStep 1 - Find max: {m.item()}")

shifted = large_logits - m[:, None]
print(f"Step 2 - Subtract max from all values: {shifted.tolist()}")
print("         Now all values are <= 0! (0, -1, -2)")

exp_shifted = shifted.exp()
print(f"Step 3 - Compute exp of shifted values: {[round(x, 4) for x in exp_shifted.squeeze().tolist()]}")
print("         exp(0)=1, exp(-1)=0.37, exp(-2)=0.14 - no overflow!")

sum_exp = exp_shifted.sum()
print(f"Step 4 - Sum the exponentials: {sum_exp.item():.4f}")

log_sum = sum_exp.log()
print(f"Step 5 - Take log of sum: {log_sum.item():.4f}")

result = m.item() + log_sum.item()
print(f"Step 6 - Add max back: {m.item()} + {log_sum.item():.4f} = {result:.4f}")

# Using our function
final_result = logsumexp(large_logits)
print(f"\nFinal result using logsumexp(): {final_result.item():.4f}")

In [ ]:
# ============================================================================
# FINAL LOG-SOFTMAX USING PYTORCH'S LOGSUMEXP
# ============================================================================
# PyTorch already has this numerically stable logsumexp built in!
# We don't need to implement it ourselves - just use x.logsumexp()

def log_softmax(x):
    """
    Compute log-softmax using PyTorch's numerically stable logsumexp.

    Formula: log_softmax(x) = x - logsumexp(x)

    Parameters:
    -----------
    x : tensor
        Input tensor of logits, shape (batch_size, num_classes)

    Returns:
    --------
    tensor
        Log-probabilities, shape (batch_size, num_classes)

    x.logsumexp(-1, keepdim=True):
        - Computes log(sum(exp(x))) along last dimension (-1 means last)
        - keepdim=True preserves the dimension for broadcasting
        - Uses the numerically stable trick internally
    """
    return x - x.logsumexp(-1, keepdim=True)

# Verify our logsumexp matches PyTorch's built-in version
test_close(logsumexp(pred), pred.logsumexp(-1), eps=1e-4)
print("Our logsumexp matches PyTorch's built-in version!")

# Apply log_softmax to our model predictions
sm_pred = log_softmax(pred)

print(f"\nPredictions shape: {pred.shape}")
print(f"Log-softmax output shape: {sm_pred.shape}")
print("\nFirst prediction (raw logits from model):")
print(pred[0])
print("\nAfter log_softmax (log-probabilities):")
print(sm_pred[0])

print("\n" + "="*60)
print("KEY OBSERVATIONS:")
print("="*60)
print("1. All log-probabilities are NEGATIVE")
print("   (because log(x) < 0 when 0 < x < 1)")
print("2. They don't sum to 1, but exp(them) does!")
print(f"   Sum of exp(log_probs) = {sm_pred[0].exp().sum():.4f}")
print("3. The largest value is closest to 0 (highest probability)")

### Step 2: Negative Log Likelihood (NLL) Loss

Now that we have log-probabilities, we need a loss function. For classification, we use **Negative Log Likelihood (NLL)** loss.

**The idea is simple:**
- We want the probability of the correct class to be high
- High probability = Low negative log probability = Low loss
- The model is penalized for assigning low probability to the correct class

**Cross-entropy loss formula:**
$$L = -\sum_i y_i \log(p_i)$$

But since our labels are **integers** (not one-hot encoded), this simplifies to:
$$L = -\log(p_{\text{correct class}})$$

We just need the log-probability of the correct class!

**Example:**
- True label is 5 (the digit five)
- Log-softmax output is `[-2.3, -2.1, -1.8, -2.5, -2.0, -0.5, ...]`
- The log-prob for class 5 is -0.5
- Loss = -(-0.5) = 0.5

The lower the loss, the more confident the model is about the correct class.

In [ ]:
# ============================================================================
# UNDERSTANDING INTEGER ARRAY INDEXING
# ============================================================================
# We need to select the log-probability of the CORRECT class for each sample.
# PyTorch (like NumPy) supports powerful integer array indexing that makes
# this very efficient.

# First, let's look at the first few true labels
print("First 3 true labels:", y_train[:3].tolist())
print("This means:")
print("  - First image is digit 5")
print("  - Second image is digit 0")
print("  - Third image is digit 4")

print("\nFirst 3 rows of log-softmax predictions (10 values each):")
print(sm_pred[:3])
print("\nEach row has 10 values - log-probabilities for classes 0-9")

In [ ]:
# ============================================================================
# MANUAL INDEXING TO GET CORRECT CLASS LOG-PROBABILITIES
# ============================================================================
# For each sample, we need the log-prob at the index of the true label
#
# Sample 0: True label is 5 -> We want sm_pred[0, 5]
# Sample 1: True label is 0 -> We want sm_pred[1, 0]
# Sample 2: True label is 4 -> We want sm_pred[2, 4]

print("Getting log-probabilities of correct classes manually:")
print(f"  Sample 0, correct class {y_train[0].item()}: log_prob = {sm_pred[0, 5]:.4f}")
print(f"  Sample 1, correct class {y_train[1].item()}: log_prob = {sm_pred[1, 0]:.4f}")
print(f"  Sample 2, correct class {y_train[2].item()}: log_prob = {sm_pred[2, 4]:.4f}")

# These are the log-probabilities the model assigns to the correct classes
# Remember: log-probs are negative, closer to 0 = higher probability

In [ ]:
# ============================================================================
# FANCY INDEXING: SELECT MULTIPLE ELEMENTS AT ONCE!
# ============================================================================
# Instead of manually getting each value one by one, PyTorch supports
# "fancy indexing" (also called advanced indexing) to get them all at once!

# sm_pred[[0,1,2], [5,0,4]] means:
#   - From row 0, take column 5
#   - From row 1, take column 0
#   - From row 2, take column 4
# The two lists pair up element-wise: (0,5), (1,0), (2,4)

# The rows are [0, 1, 2] and columns are the true labels [5, 0, 4]
row_indices = [0, 1, 2]
col_indices = y_train[:3].tolist()  # [5, 0, 4]

result = sm_pred[row_indices, col_indices]

print("Fancy indexing:")
print(f"  Row indices: {row_indices}")
print(f"  Column indices (true labels): {col_indices}")
print(f"  Result (log-probs of correct classes): {result}")

print("\nThis is the SAME as our manual indexing above!")
print("Fancy indexing is MUCH more efficient than looping.")

# For all samples: range(n) gives row indices, y_train gives column indices
# This gets the log-prob of the correct class for EVERY sample at once!

In [ ]:
# ============================================================================
# NEGATIVE LOG LIKELIHOOD (NLL) LOSS FUNCTION
# ============================================================================

def nll(input, target):
    """
    Compute Negative Log Likelihood loss.

    Parameters:
    -----------
    input : tensor
        Log-probabilities from log_softmax, shape (batch_size, num_classes)
    target : tensor
        True class labels as integers, shape (batch_size,)

    Returns:
    --------
    tensor
        Scalar loss value (average NLL across the batch)

    How it works:
    1. range(target.shape[0]) creates [0, 1, 2, ..., batch_size-1]
       These are the row indices (one per sample)
    2. target contains the column indices (the true class for each sample)
    3. input[row_indices, col_indices] gets log-prob of correct class for each sample
    4. .mean() averages across the batch
    5. Negate because we want NEGATIVE log likelihood (loss should be positive)

    Example:
        input[0] = [-2.3, -1.5, -0.5, ...] and target[0] = 2
        We select input[0, 2] = -0.5
        Negate to get 0.5
    """
    # Get row indices: [0, 1, 2, ..., batch_size-1]
    batch_indices = range(target.shape[0])

    # Get log-prob of correct class for each sample
    correct_class_log_probs = input[batch_indices, target]

    # Average and negate
    return -correct_class_log_probs.mean()

print("NLL (Negative Log Likelihood) loss function defined!")
print("\nFormula: loss = -mean(log_prob[correct_class])")
print("\nWhy negative?")
print("  - log(probability) is always <= 0 (since probability <= 1)")
print("  - We want LOSS to be positive")
print("  - Lower loss = higher probability on correct class = better!")

In [ ]:
# ============================================================================
# COMPUTE NLL LOSS ON OUR PREDICTIONS
# ============================================================================

loss = nll(sm_pred, y_train)

print(f"NLL Loss on training data: {loss:.4f}")
print("\n" + "="*60)
print("INTERPRETING THE LOSS VALUE:")
print("="*60)
print("- Lower loss is better (high probability on correct class)")
print("- Perfect model: loss = 0 (100% confident and correct)")
print("- Random guessing with 10 classes: loss = -log(0.1) = 2.30")
print(f"  (because probability would be ~10% for each class)")
print(f"\nOur untrained model has loss = {loss:.4f}")
print("This is about what we'd expect for random predictions.")

### Verifying Our Implementation Against PyTorch

Let's verify that our implementation matches PyTorch's built-in functions. PyTorch provides:
- `F.log_softmax()`: Computes log-softmax
- `F.nll_loss()`: Computes NLL loss (expects log-probabilities as input)
- `F.cross_entropy()`: Combines both into one function (expects raw logits as input)

In [ ]:
# ============================================================================
# VERIFY AGAINST PYTORCH'S F.nll_loss
# ============================================================================
# F.nll_loss expects:
#   - Log-probabilities (from log_softmax)
#   - Integer targets
# It computes the same thing as our nll() function

# Our implementation
our_loss = loss

# PyTorch's implementation
# F.log_softmax(pred, -1) computes log_softmax along last dimension
# F.nll_loss then computes NLL from these log-probs
pytorch_loss = F.nll_loss(F.log_softmax(pred, -1), y_train)

# Check they're the same
test_close(pytorch_loss, our_loss, 1e-3)
print("Our NLL loss matches PyTorch's F.nll_loss!")
print(f"  Our loss:     {our_loss:.6f}")
print(f"  PyTorch loss: {pytorch_loss:.6f}")

### The F.cross_entropy Shortcut

PyTorch combines `log_softmax` and `nll_loss` into one optimized function: `F.cross_entropy`.

**Important**: `F.cross_entropy` takes raw **logits** (not log-probabilities)!

```python
# These are equivalent:
F.cross_entropy(logits, targets)
F.nll_loss(F.log_softmax(logits, dim=-1), targets)
```

Using `F.cross_entropy` is:
- More efficient (combines operations)
- More numerically stable
- Less code to write

In [ ]:
# ============================================================================
# VERIFY AGAINST F.cross_entropy
# ============================================================================
# F.cross_entropy = log_softmax + nll_loss combined
# It takes RAW LOGITS (not log-probabilities)

cross_entropy_loss = F.cross_entropy(pred, y_train)

test_close(cross_entropy_loss, loss, 1e-3)
print("F.cross_entropy(logits, targets) matches our loss!")
print(f"  Our loss:           {loss:.6f}")
print(f"  F.cross_entropy:    {cross_entropy_loss:.6f}")

print("\n" + "="*60)
print("SUMMARY: From now on, we'll use F.cross_entropy")
print("="*60)
print("Advantages:")
print("  - Takes raw logits (model output) directly")
print("  - Combines log_softmax + nll_loss efficiently")
print("  - More numerically stable")
print("  - Standard practice in PyTorch

---

## Part 3: The Basic Training Loop

Now we have all the pieces to train a neural network! Let's put it all together.

### What is a Training Loop?

A training loop is the process of:
1. **Get a minibatch** of data (e.g., 50 images)
2. **Forward pass**: Run data through the model to get predictions
3. **Compute loss**: Measure how wrong the predictions are
4. **Backward pass**: Compute gradients (how to change each weight)
5. **Update weights**: Adjust weights in the direction that reduces loss
6. **Repeat** for all minibatches (that's one epoch)
7. **Repeat** for multiple epochs

```
+-----------------------------------------------------------------------------+
|                     MINIBATCH TRAINING LOOP                                 |
+-----------------------------------------------------------------------------+
|                                                                             |
|   FOR epoch = 1 to num_epochs:                                              |
|   |                                                                         |
|   |   FOR i = 0, bs, 2*bs, 3*bs, ..., n:    (step through data in batches) |
|   |   |                                                                     |
|   |   |   xb, yb = data[i:i+bs]             (get one minibatch)            |
|   |   |                                                                     |
|   |   |   preds = model(xb)                 (forward pass)                 |
|   |   |   loss = loss_func(preds, yb)       (compute loss)                 |
|   |   |   loss.backward()                   (compute gradients)            |
|   |   |                                                                     |
|   |   |   FOR each parameter p:             (update weights)               |
|   |   |   |   p = p - learning_rate * p.grad                               |
|   |   |   |   p.grad.zero_()                (clear gradients for next step)|
|   |   |                                                                     |
|   |   Print epoch statistics                                                |
|                                                                             |
+-----------------------------------------------------------------------------+
```

In [ ]:
# ============================================================================
# SETUP FOR TRAINING
# ============================================================================

# Use cross-entropy as our loss function
# This takes raw logits and integer targets, and computes log_softmax + nll internally
loss_func = F.cross_entropy

print("Loss function: F.cross_entropy")
print("  - Takes raw logits (model output) and integer targets")
print("  - Combines log_softmax + nll_loss efficiently")
print("  - This is the standard loss for classification problems")

### Getting a Minibatch

Instead of processing all 50,000 images at once (which would use too much memory), we process small batches. Let's see how to extract a minibatch from our data.

In [ ]:
# ============================================================================
# GETTING A MINIBATCH
# ============================================================================
# Instead of processing all 50,000 images at once, we process small batches.
# This is more memory-efficient and actually helps training!

bs = 50  # batch size - how many samples per minibatch
         # Common batch sizes: 16, 32, 64, 128, 256
         # Larger = faster training, more memory, less noise
         # Smaller = slower training, less memory, more noise

# Get the first minibatch using Python slicing
# x_train[0:50] = first 50 images
# y_train[0:50] = first 50 labels
xb = x_train[0:bs]  # Input batch, shape (50, 784)
yb = y_train[0:bs]  # Target batch, shape (50,)

print(f"Batch size: {bs}")
print(f"\nInput batch (xb):")
print(f"  Shape: {xb.shape}")
print(f"  This is {bs} images, each with {xb.shape[1]} pixels")
print(f"\nTarget batch (yb):")
print(f"  Shape: {yb.shape}")
print(f"  These are the true labels (integers 0-9)")
print(f"  First few labels: {yb[:10].tolist()}")

# Get predictions for this batch
preds = model(xb)

print(f"\nPredictions:")
print(f"  Shape: {preds.shape}")
print(f"  Each image has 10 output values (logits, one per class)")

In [ ]:
# ============================================================================
# LOOKING AT OUR TARGETS (TRUE LABELS)
# ============================================================================
# These are the correct digit labels for our minibatch

print("True labels for the first batch of 50 images:")
print(yb)
print("\nThese are integers from 0-9, each representing a digit.")
print("\nDistribution of digits in this batch:")
for digit in range(10):
    count = (yb == digit).sum().item()
    print(f"  Digit {digit}: {count} images")

In [ ]:
# ============================================================================
# COMPUTE LOSS FOR ONE MINIBATCH
# ============================================================================

batch_loss = loss_func(preds, yb)

print(f"Loss for this minibatch: {batch_loss:.4f}")
print("\nThis single number tells us how wrong our predictions are")
print("for all 50 images combined (averaged).")
print("\nLower loss = better predictions = model is learning!")

In [ ]:
# ============================================================================
# GETTING PREDICTED CLASSES FROM LOGITS
# ============================================================================
# The model outputs 10 logits per image. To get the predicted class,
# we take argmax - the index of the maximum value!

# argmax(dim=1) finds the index of max value along dimension 1 (the classes)
# dim=0 would be along the batch, dim=1 is along the classes
predicted_classes = preds.argmax(dim=1)

print("The model outputs 10 logits (one per class 0-9).")
print("The predicted class is whichever has the highest logit.")
print("\nExample for first image:")
print(f"  Logits: {preds[0].detach()}")
print(f"  Max value: {preds[0].max().item():.2f} at index {preds[0].argmax().item()}")
print(f"  Predicted class: {predicted_classes[0].item()}")
print(f"  True class: {yb[0].item()}")

print("\n" + "-"*50)
print("Predicted classes (first 10):", predicted_classes[:10].tolist())
print("True labels (first 10):      ", yb[:10].tolist())
print("\nMatches:", (predicted_classes[:10] == yb[:10]).sum().item(), "out of 10")

### Accuracy: A More Intuitive Metric

While loss tells us how wrong we are (mathematically), **accuracy** is more intuitive - it's simply the percentage of correct predictions.

In [ ]:
# ============================================================================
# ACCURACY FUNCTION
# ============================================================================
#|export

def accuracy(out, yb):
    """
    Compute classification accuracy.

    Parameters:
    -----------
    out : tensor
        Model output logits, shape (batch_size, num_classes)
    yb : tensor
        True labels as integers, shape (batch_size,)

    Returns:
    --------
    tensor
        Accuracy as a float between 0.0 and 1.0 (e.g., 0.95 = 95%)

    How it works step by step:
        out.argmax(dim=1)  -> Get predicted class for each sample
                             Returns tensor of shape (batch_size,)
        == yb              -> Compare to true labels (element-wise)
                             Returns boolean tensor (True/False for each)
        .float()           -> Convert True=1.0, False=0.0
        .mean()            -> Average to get proportion correct
    """
    return (out.argmax(dim=1) == yb).float().mean()

# Let's trace through this step by step
print("How accuracy() works step by step:")
print("="*50)

# Step 1: Get predictions
pred_classes = preds.argmax(dim=1)
print(f"1. Predicted classes (first 10): {pred_classes[:10].tolist()}")

# Step 2: Compare to true labels
correct = pred_classes == yb
print(f"2. Correct? (first 10): {correct[:10].tolist()}")

# Step 3: Convert to float
correct_float = correct.float()
print(f"3. As floats (first 10): {correct_float[:10].tolist()}")

# Step 4: Average
acc = correct_float.mean()
print(f"4. Average = {acc:.4f}")

print(f"\nUsing accuracy(): {accuracy(preds, yb):.4f}")
print(f"This means {accuracy(preds, yb)*100:.1f}% of predictions are correct")

In [ ]:
# ============================================================================
# TEST ACCURACY ON INITIAL (RANDOM) MODEL
# ============================================================================

initial_accuracy = accuracy(preds, yb)

print(f"Initial accuracy (random weights): {initial_accuracy:.2%}")
print("\nExpected accuracy for random guessing:")
print(f"  With 10 equally likely classes: 1/10 = 10%")
print(f"  Our model: {initial_accuracy:.1%}")
print("\nThe model is performing at roughly chance level - as expected!")
print("Training should improve this significantly.")

### Hyperparameters: Values We Choose

**Hyperparameters** are settings WE choose before training - they're not learned by the model. Two important ones:

- **Learning rate (lr)**: How big a step to take when updating weights
  - Too high: Training becomes unstable, loss may increase
  - Too low: Training is very slow
  - Typical values: 0.001 to 1.0 (depends on the problem)

- **Number of epochs**: How many times to loop through all training data
  - More epochs = more training = (usually) better performance
  - But too many epochs can lead to overfitting

In [ ]:
# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================

lr = 0.5    # Learning rate - how big a step when updating weights
            # We'll discuss how to choose this later
            # 0.5 works well for this simple problem

epochs = 3  # Number of times to iterate through the entire training set
            # Each epoch = model sees all 50,000 training images once

print("Hyperparameters:")
print(f"  Learning rate (lr): {lr}")
print(f"  Number of epochs: {epochs}")
print("\nThese are hyperparameters - WE choose them, they're not learned.")
print("Finding good hyperparameters is an important part of deep learning!")

In [ ]:
# ============================================================================
# REPORT FUNCTION - PRINT LOSS AND ACCURACY
# ============================================================================
#|export

def report(loss, preds, yb):
    """
    Print loss and accuracy for a batch.

    Parameters:
    -----------
    loss : tensor
        The loss value (scalar)
    preds : tensor
        Model predictions (logits)
    yb : tensor
        True labels

    Prints:
        loss, accuracy (both with 2 decimal places)
    """
    print(f'{loss:.2f}, {accuracy(preds, yb):.2f}')

print("report() function defined!")
print("Usage: report(loss, predictions, targets)")
print("Output format: loss, accuracy")

In [ ]:
# ============================================================================
# CHECK INITIAL PERFORMANCE (BEFORE TRAINING)
# ============================================================================

# Get a fresh batch and predictions
xb, yb = x_train[:bs], y_train[:bs]
preds = model(xb)

print("Initial performance (random weights):")
print("Format: loss, accuracy")
report(loss_func(preds, yb), preds, yb)
print("\nHigh loss (~2.3) and low accuracy (~10%) - expected for random model!")
print("Cross-entropy loss for random guessing = -log(0.1) = 2.30")

### The Complete Training Loop

Now let's put it all together! This is the core of neural network training.

In [ ]:
# ============================================================================
# THE BASIC TRAINING LOOP
# ============================================================================
# This is the core of neural network training!

print("="*60)
print("TRAINING THE MODEL")
print("="*60)
print(f"Training samples: {n:,}")
print(f"Batch size: {bs}")
print(f"Batches per epoch: {n // bs:,}")
print(f"Epochs: {epochs}")
print("\nTraining...")
print("Format: loss, accuracy")
print("-"*30)

for epoch in range(epochs):
    # =========================================================================
    # Loop through the data in minibatches
    # =========================================================================
    # range(0, n, bs) gives: 0, 50, 100, 150, ..., 49950
    # This steps through the data in chunks of size bs

    for i in range(0, n, bs):
        # Create a slice object for this batch: [i:i+bs]
        # slice(0, 50), slice(50, 100), slice(100, 150), etc.
        # min(n, i+bs) handles the last batch which might be smaller
        s = slice(i, min(n, i+bs))

        # Get the minibatch
        xb, yb = x_train[s], y_train[s]

        # ---------------------------------------------------------------------
        # STEP 1: FORWARD PASS
        # ---------------------------------------------------------------------
        # Run the batch through the model to get predictions
        preds = model(xb)

        # ---------------------------------------------------------------------
        # STEP 2: COMPUTE LOSS
        # ---------------------------------------------------------------------
        # How wrong are our predictions?
        loss = loss_func(preds, yb)

        # ---------------------------------------------------------------------
        # STEP 3: BACKWARD PASS
        # ---------------------------------------------------------------------
        # Compute gradients - this calculates d(loss)/d(weight) for EVERY weight
        # PyTorch does this automatically using the chain rule!
        loss.backward()

        # ---------------------------------------------------------------------
        # STEP 4: UPDATE WEIGHTS (Gradient Descent)
        # ---------------------------------------------------------------------
        # torch.no_grad() tells PyTorch not to track these operations
        # (we don't need gradients of the weight updates themselves)
        with torch.no_grad():
            # Loop through all layers in our model
            for l in model.layers:
                # Only update layers that have weights (Linear layers have weights,
                # ReLU doesn't)
                if hasattr(l, 'weight'):
                    # GRADIENT DESCENT: new_weight = old_weight - lr * gradient
                    # The gradient points "uphill" (direction of increasing loss)
                    # We go the opposite direction (subtract) to decrease loss
                    l.weight -= l.weight.grad * lr
                    l.bias -= l.bias.grad * lr

                    # -------------------------------------------------------------
                    # STEP 5: ZERO GRADIENTS
                    # -------------------------------------------------------------
                    # CRITICAL: Zero the gradients for next iteration!
                    # Gradients ACCUMULATE in PyTorch, so without this,
                    # the new gradients would be added to the old ones
                    l.weight.grad.zero_()
                    l.bias.grad.zero_()

    # Report at end of each epoch
    report(loss, preds, yb)

print("-"*30)
print("Training complete!")
print("\nNotice how loss DECREASED and accuracy INCREASED each epoch!")
print("The model is learning to classify digits.")

### What Just Happened?

Let's break down the training loop:

**Each Iteration (One Minibatch):**
1. **Forward pass** (`preds = model(xb)`): Data flows through the network, producing predictions
2. **Compute loss** (`loss = loss_func(preds, yb)`): Single number measuring total error
3. **Backward pass** (`loss.backward()`): PyTorch computes gradients for ALL parameters
4. **Update weights** (`weight -= lr * gradient`): Move weights to reduce loss
5. **Zero gradients** (`grad.zero_()`): Clear gradients for next iteration

**Why Zero Gradients?**
- PyTorch ACCUMULATES gradients by default (adds new to old)
- This is useful for some advanced techniques
- But for standard training, we want fresh gradients each iteration
- Forgetting to zero gradients is a common bug!

**What's Gradient Descent?**
- The gradient tells us which direction increases the loss
- We go the OPPOSITE direction to decrease the loss
- Learning rate controls how big a step we take
- `new_weight = old_weight - learning_rate * gradient`

---

## Part 4: Using nn.Module.parameters() and Optimizers

The training loop above works, but it has several problems:
1. We manually loop through layers to find weights
2. We manually update each weight
3. We manually zero gradients

This is tedious and error-prone! PyTorch provides better abstractions:
- `nn.Module.parameters()`: Automatically finds all learnable parameters
- `torch.optim`: Built-in optimizers that handle weight updates

```
+-----------------------------------------------------------------------------+
|                    MANUAL vs PyTorch APPROACH                               |
+-----------------------------------------------------------------------------+
|                                                                             |
|   MANUAL (what we did):                                                     |
|   ---------------------                                                     |
|   for l in model.layers:                                                    |
|       if hasattr(l, 'weight'):                                              |
|           l.weight -= l.weight.grad * lr                                    |
|           l.bias -= l.bias.grad * lr                                        |
|           l.weight.grad.zero_()                                             |
|           l.bias.grad.zero_()                                               |
|                                                                             |
|   PYTORCH (cleaner):                                                        |
|   ------------------                                                        |
|   opt.step()       # Update all parameters                                  |
|   opt.zero_grad()  # Zero all gradients                                     |
|                                                                             |
+-----------------------------------------------------------------------------+
```

### The Problem with Manual Layer Lists

In our previous model, we stored layers in a Python list:

```python
self.layers = [nn.Linear(784, 50), nn.ReLU(), nn.Linear(50, 10)]
```

The problem: **PyTorch doesn't know these are part of the model!** When layers are just in a Python list, PyTorch can't automatically find their parameters.

**The solution:** Store layers as attributes of the nn.Module, or use special containers like `nn.ModuleList` or `nn.Sequential`.

In [ ]:
# ============================================================================
# DEMONSTRATION: nn.Module TRACKS ATTRIBUTES
# ============================================================================
# When you assign a layer as an ATTRIBUTE of nn.Module, PyTorch tracks it!

# Create an empty module and add a layer as an attribute
m1 = nn.Module()
m1.foo = nn.Linear(3, 4)  # 'foo' is the attribute name

print("Module with one layer as attribute:")
print(m1)

print("\n" + "="*50)
print("Child modules (layers that are attributes):")
print(list(m1.named_children()))

print("\nParameters found automatically:")
for name, param in m1.named_parameters():
    print(f"  {name}: shape {param.shape}")

In [ ]:
# ============================================================================
# MLP CLASS WITH PROPER ATTRIBUTE REGISTRATION
# ============================================================================

class MLP(nn.Module):
    """
    Multi-Layer Perceptron with proper attribute registration.

    By storing layers as ATTRIBUTES (self.l1 = ...) instead of in a list,
    PyTorch automatically tracks them and can find their parameters!
    """
    def __init__(self, n_in, nh, n_out):
        super().__init__()
        # These are ATTRIBUTES, not items in a list
        # PyTorch will automatically register them
        self.l1 = nn.Linear(n_in, nh)    # First linear layer
        self.l2 = nn.Linear(nh, n_out)   # Second linear layer
        self.relu = nn.ReLU()            # Activation (no parameters)

    def forward(self, x):
        """Forward pass - standard method name in PyTorch.""        return self.l2(self.relu(self.l1(x)))

# Create model
model = MLP(m, nh, 10)

print("MLP model created:")
print(model)

print("\n" + "="*50)
print("Child modules (automatically tracked!):")
for name, module in model.named_children():
    print(f"  {name}: {module}")

print("\nParameters (automatically found!):")
for param in model.parameters():
    print(f"  Shape: {param.shape}, requires_grad: {param.requires_grad}")

### Using model.parameters() in Training

Now that PyTorch can find our parameters automatically, we can simplify the training loop.

In [ ]:
# ============================================================================
# SIMPLIFIED TRAINING WITH model.parameters()
# ============================================================================

def fit():
    """Train the model using model.parameters() to find weights.""    for epoch in range(epochs):
        for i in range(0, n, bs):
            s = slice(i, min(n, i+bs))
            xb, yb = x_train[s], y_train[s]
            preds = model(xb)
            loss = loss_func(preds, yb)
            loss.backward()

            # SIMPLIFIED: Loop through ALL parameters automatically!
            with torch.no_grad():
                for p in model.parameters():
                    p -= p.grad * lr  # Update parameter

                # Zero ALL gradients at once
                model.zero_grad()

        report(loss, preds, yb)

# Train!
print("Training with model.parameters()...")
print("Format: loss, accuracy")
fit()
print("\nMuch cleaner than manually finding weights!")

### How nn.Module Tracks Submodules

Behind the scenes, PyTorch overrides the `__setattr__` method to automatically register submodules. Here's a simplified version showing how it works:

In [ ]:
# ============================================================================
# HOW nn.Module TRACKS SUBMODULES (SIMPLIFIED)
# ============================================================================
# This shows the magic behind PyTorch's parameter tracking

class MyModule:
    """Simplified version showing how nn.Module tracks submodules.""    def __init__(self, n_in, nh, n_out):
        # Dictionary to store child modules
        self._modules = {}

        # When we assign attributes, __setattr__ is called
        # which adds them to _modules
        self.l1 = nn.Linear(n_in, nh)
        self.l2 = nn.Linear(nh, n_out)

    def __setattr__(self, k, v):
        """Called whenever an attribute is assigned (self.foo = bar).""        # If not a private attribute (starting with _), register it
        if not k.startswith("_"):
            self._modules[k] = v
        # Actually set the attribute
        super().__setattr__(k, v)

    def __repr__(self):
        return f'{self._modules}'

    def parameters(self):
        """Yield all parameters from all registered modules.""        for module in self._modules.values():
            yield from module.parameters()

# Test it
mdl = MyModule(m, nh, 10)
print("Registered modules:", mdl)
print("\nParameters found:")
for p in mdl.parameters():
    print(f"  {p.shape}")

### nn.ModuleList and nn.Sequential

Sometimes we want to keep layers in a list. PyTorch provides special containers for this:

- **nn.ModuleList**: A list that properly registers all modules
- **nn.Sequential**: Like ModuleList, but also handles the forward pass automatically

In [ ]:
# ============================================================================
# nn.ModuleList - A LIST THAT REGISTERS MODULES
# ============================================================================

layers = [nn.Linear(m, nh), nn.ReLU(), nn.Linear(nh, 10)]

class SequentialModel(nn.Module):
    def __init__(self, layers):
        super().__init__()
        # nn.ModuleList properly registers all layers!
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = SequentialModel(layers)
print("Model with nn.ModuleList:")
print(model)

print("\nAll parameters are properly tracked:")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
# ============================================================================
# nn.Sequential - THE SIMPLEST WAY
# ============================================================================
# nn.Sequential is even simpler - it creates the forward() method for you!

# Just pass the layers in order
model = nn.Sequential(
    nn.Linear(m, nh),   # 784 -> 50
    nn.ReLU(),          # Activation
    nn.Linear(nh, 10)   # 50 -> 10
)

print("Model using nn.Sequential:")
print(model)

print("\nThis is the cleanest way to define simple sequential models!")
print("nn.Sequential automatically:")
print("  - Registers all modules")
print("  - Creates forward() method (passes through layers in order)")

# Test it
print(f"\nTest forward pass shape: {model(xb).shape}")

In [ ]:
# ============================================================================
# TRAIN WITH nn.Sequential MODEL
# ============================================================================
print("Training nn.Sequential model...")
print("Format: loss, accuracy")
fit()

# Test final performance
print(f"\nFinal check: loss={loss_func(model(xb), yb):.4f}, accuracy={accuracy(model(xb), yb):.2%}")

### Building Our Own Optimizer

Now let's encapsulate the weight update logic into an **Optimizer** class. This makes our training loop even cleaner.

In [ ]:
# ============================================================================
# BUILDING AN OPTIMIZER CLASS
# ============================================================================

class Optimizer:
    """
    Simple optimizer that performs gradient descent.

    An optimizer handles:
    1. Storing references to parameters
    2. Updating parameters (step)
    3. Zeroing gradients (zero_grad)
    """
    def __init__(self, params, lr=0.5):
        """
        Initialize the optimizer.

        Parameters:
        -----------
        params : iterable
            Model parameters to optimize (from model.parameters())
        lr : float
            Learning rate (step size for updates)
        """
        self.params = list(params)  # Store as list (generators exhaust)
        self.lr = lr

    def step(self):
        """Update all parameters using gradient descent.""        with torch.no_grad():
            for p in self.params:
                p -= p.grad * self.lr  # p = p - lr * gradient

    def zero_grad(self):
        """Zero all gradients.""        for p in self.params:
            p.grad.data.zero_()

print("Optimizer class defined!")
print("\nMethods:")
print("  - step(): Update all parameters")
print("  - zero_grad(): Clear all gradients")

In [ ]:
# ============================================================================
# TRAINING WITH OUR OPTIMIZER
# ============================================================================

# Create fresh model
model = nn.Sequential(nn.Linear(m, nh), nn.ReLU(), nn.Linear(nh, 10))

# Create optimizer
opt = Optimizer(model.parameters(), lr=lr)

print("Training with our Optimizer class...")
print("Format: loss, accuracy")

for epoch in range(epochs):
    for i in range(0, n, bs):
        s = slice(i, min(n, i+bs))
        xb, yb = x_train[s], y_train[s]
        preds = model(xb)
        loss = loss_func(preds, yb)
        loss.backward()

        # Just two lines for update + zero_grad!
        opt.step()
        opt.zero_grad()

    report(loss, preds, yb)

print("\nMuch cleaner training loop!")

### PyTorch's Built-in Optimizers

PyTorch provides `torch.optim.SGD` which does exactly what our optimizer does, plus more advanced features like momentum (which we'll cover later).

Common optimizers in `torch.optim`:
- **SGD**: Stochastic Gradient Descent (what we implemented)
- **Adam**: Adaptive learning rates (very popular)
- **AdamW**: Adam with proper weight decay
- **RMSprop**: Another adaptive method

In [ ]:
# ============================================================================
# USING PYTORCH'S BUILT-IN OPTIMIZER
# ============================================================================

from torch import optim  # PyTorch's optimization module

def get_model():
    """Helper function to create a fresh model and optimizer.""    model = nn.Sequential(nn.Linear(m, nh), nn.ReLU(), nn.Linear(nh, 10))
    # optim.SGD is PyTorch's Stochastic Gradient Descent
    # It has additional features like momentum, weight_decay, etc.
    opt = optim.SGD(model.parameters(), lr=lr)
    return model, opt

# Create model and optimizer
model, opt = get_model()

print("Using torch.optim.SGD:")
print(f"  Learning rate: {lr}")
print(f"  Parameters being optimized: {len(list(model.parameters()))}")

# Check initial loss
print(f"\nInitial loss: {loss_func(model(xb), yb):.4f}")

In [ ]:
# ============================================================================
# THE STANDARD PYTORCH TRAINING LOOP
# ============================================================================
# This is the standard pattern you'll see everywhere in PyTorch!

print("Standard PyTorch training loop:")
print("Format: loss, accuracy")

for epoch in range(epochs):
    # Iterate through training data in mini-batches
    for i in range(0, n, bs):
        # Get batch
        s = slice(i, min(n, i + bs))
        xb, yb = x_train[s], y_train[s]

        # Forward pass
        preds = model(xb)

        # Compute loss
        loss = loss_func(preds, yb)

        # Backward pass
        loss.backward()

        # Update weights
        opt.step()

        # Zero gradients
        opt.zero_grad()

    # Report at end of epoch
    report(loss, preds, yb)

print("\nThis is THE standard training loop pattern in PyTorch!")

---

## Part 5: Dataset and DataLoader

Our training loop still has clunky code for getting batches:

```python
for i in range(0, n, bs):
    xb, yb = x_train[i:min(n,i+bs)], y_train[i:min(n,i+bs)]
```

PyTorch provides elegant abstractions for data loading:
- **Dataset**: Represents a collection of samples with `__getitem__` and `__len__`
- **DataLoader**: Handles batching, shuffling, and parallel loading

Let's build these from scratch to understand how they work!

### Building a Dataset Class

A Dataset just needs two methods:
- `__len__`: How many samples are there?
- `__getitem__`: Get one or more samples by index

This simple interface allows uniform data handling regardless of where data comes from (memory, files, databases, etc.).

In [ ]:
# ============================================================================
# DATASET CLASS
# ============================================================================
#|export

class Dataset:
    """
    A simple Dataset class that pairs inputs (x) with targets (y).

    A Dataset provides:
    - __len__: Total number of samples
    - __getitem__: Access samples by index (supports slicing!)

    This abstraction lets us handle data uniformly, regardless of whether
    it's in memory, on disk, in a database, etc.
    """
    def __init__(self, x, y):
        """Store the data tensors.""        self.x = x  # Input data (e.g., images)
        self.y = y  # Target labels

    def __len__(self):
        """Return the number of samples.""        return len(self.x)

    def __getitem__(self, i):
        """
        Get sample(s) by index.

        Parameters:
        -----------
        i : int or slice
            Index or slice to retrieve

        Returns:
        --------
        tuple
            (input, target) for the requested sample(s)
        """
        return self.x[i], self.y[i]

print("Dataset class defined!")
print("\nKey methods:")
print("  __len__():      Returns number of samples")
print("  __getitem__(i): Returns (x[i], y[i])")

In [ ]:
# ============================================================================
# CREATE TRAINING AND VALIDATION DATASETS
# ============================================================================

train_ds = Dataset(x_train, y_train)
valid_ds = Dataset(x_valid, y_valid)

# Verify the datasets
print("Datasets created!")
print(f"\nTraining dataset:")
print(f"  Length: {len(train_ds)}")
print(f"  First sample shapes: x={train_ds[0][0].shape}, y={train_ds[0][1].shape}")

print(f"\nValidation dataset:")
print(f"  Length: {len(valid_ds)}")

# Test slicing
xb, yb = train_ds[0:5]  # Get first 5 samples
print(f"\nSlicing works! train_ds[0:5] gives:")
print(f"  xb shape: {xb.shape}")
print(f"  yb shape: {yb.shape}")

# Verify shapes match our expectations
assert len(train_ds) == len(x_train), "Training dataset length mismatch"
assert len(valid_ds) == len(x_valid), "Validation dataset length mismatch"
print("\nAll assertions passed!")

In [ ]:
# ============================================================================
# TRAINING WITH DATASET
# ============================================================================
# Now we can use train_ds[i:j] instead of x_train[i:j], y_train[i:j]

model, opt = get_model()

print("Training with Dataset...")
print("Format: loss, accuracy")

for epoch in range(epochs):
    for i in range(0, n, bs):
        # Much cleaner! One line instead of two
        xb, yb = train_ds[i:min(n, i+bs)]
        preds = model(xb)
        loss = loss_func(preds, yb)
        loss.backward()
        opt.step()
        opt.zero_grad()
    report(loss, preds, yb)

print("\nDataset makes accessing data cleaner!")

### Building a DataLoader Class

The DataLoader handles:
- Iterating through the dataset in batches
- (Later) Shuffling the data
- (Later) Loading in parallel with multiple workers

Let's start with a simple version:

In [ ]:
# ============================================================================
# DATALOADER CLASS
# ============================================================================

class DataLoader:
    """
    Iterates through a Dataset in batches.

    A DataLoader provides a clean iteration interface:
        for xb, yb in dataloader:
            # xb and yb are batches

    Instead of:
        for i in range(0, len(ds), bs):
            xb, yb = ds[i:i+bs]
    """
    def __init__(self, ds, bs):
        """
        Parameters:
        -----------
        ds : Dataset
            The dataset to iterate over
        bs : int
            Batch size
        """
        self.ds = ds
        self.bs = bs

    def __iter__(self):
        """
        Yield batches of (x, y) from the dataset.

        This is a generator function - it yields batches one at a time.
        """
        for i in range(0, len(self.ds), self.bs):
            yield self.ds[i:i+self.bs]

print("DataLoader class defined!")
print("\nUsage:")
print("  for xb, yb in dataloader:")
print("      # process batch")

In [ ]:
# ============================================================================
# CREATE DATALOADERS
# ============================================================================

train_dl = DataLoader(train_ds, bs)
valid_dl = DataLoader(valid_ds, bs)

print("DataLoaders created!")
print(f"  Batch size: {bs}")
print(f"  Training batches: {len(train_ds) // bs}")
print(f"  Validation batches: {len(valid_ds) // bs}")

# Test iteration
xb, yb = next(iter(valid_dl))  # Get first batch
print(f"\nFirst validation batch:")
print(f"  xb shape: {xb.shape}")
print(f"  yb shape: {yb.shape}")

# Visualize one sample from the batch
import matplotlib.pyplot as plt
plt.figure(figsize=(3, 3))
plt.imshow(xb[0].view(28, 28))
plt.title(f"Label: {yb[0].item()}")
plt.axis('off')
plt.show()

In [ ]:
# ============================================================================
# CLEAN TRAINING LOOP WITH DATALOADER
# ============================================================================

model, opt = get_model()

def fit():
    """Training loop using DataLoader - the cleanest version yet!""    for epoch in range(epochs):
        # This is SO much cleaner!
        for xb, yb in train_dl:
            preds = model(xb)
            loss = loss_func(preds, yb)
            loss.backward()
            opt.step()
            opt.zero_grad()
        report(loss, preds, yb)

print("Training with DataLoader...")
print("Format: loss, accuracy")
fit()

# Check performance
print(f"\nFinal: loss={loss_func(model(xb), yb):.4f}, accuracy={accuracy(model(xb), yb):.2%}")

### Adding Random Shuffling

For training, we want to **shuffle** the data each epoch. Why?
- Prevents the model from learning the order of examples
- Helps generalization
- But we DON'T shuffle validation data (order doesn't matter for evaluation)

Let's build a Sampler class to control the order of indices:

In [ ]:
# ============================================================================
# SAMPLER CLASS - CONTROLS ORDER OF INDICES
# ============================================================================

import random

class Sampler:
    """
    Controls the order in which samples are accessed.

    Parameters:
    -----------
    ds : Dataset
        The dataset to sample from
    shuffle : bool
        If True, randomize order each iteration
    """
    def __init__(self, ds, shuffle=False):
        self.n = len(ds)      # Number of samples
        self.shuffle = shuffle

    def __iter__(self):
        """Yield indices in order (or shuffled order).""        res = list(range(self.n))  # [0, 1, 2, ..., n-1]
        if self.shuffle:
            random.shuffle(res)    # Randomize in-place
        return iter(res)

# Test samplers
print("Testing Samplers:")

# Sequential sampler (no shuffle)
ss = Sampler(train_ds, shuffle=False)
print(f"\nSequential (first 10 indices): {list(ss)[:10]}")

# Random sampler
rs = Sampler(train_ds, shuffle=True)
print(f"Random (first 10 indices): {list(rs)[:10]}")
print(f"Random again (different!): {list(Sampler(train_ds, shuffle=True))[:10]}")

In [ ]:
# ============================================================================
# BATCH SAMPLER - GROUPS INDICES INTO BATCHES
# ============================================================================
import fastcore.all as fc

class BatchSampler:
    """
    Groups indices from a Sampler into batches.

    Instead of yielding individual indices, yields lists of indices
    of size batch_size.
    """
    def __init__(self, sampler, bs, drop_last=False):
        # fc.store_attr() is a fastcore utility that does:
        # self.sampler = sampler
        # self.bs = bs
        # self.drop_last = drop_last
        fc.store_attr()

    def __iter__(self):
        """Yield batches of indices.""        # fc.chunked groups items into chunks of size bs
        yield from fc.chunked(iter(self.sampler), self.bs, drop_last=self.drop_last)

# Test batch sampler
ss = Sampler(train_ds, shuffle=True)
bs_sampler = BatchSampler(ss, bs=4)

# Get first few batches of indices
from itertools import islice
print("Batch Sampler (first 5 batches of 4 indices each):")
for i, batch in enumerate(islice(bs_sampler, 5)):
    print(f"  Batch {i}: {batch}")

In [ ]:
# ============================================================================
# COLLATE FUNCTION - COMBINES SAMPLES INTO A BATCH
# ============================================================================

def collate(batch):
    """
    Combine a list of (x, y) tuples into batch tensors.

    Parameters:
    -----------
    batch : list of tuples
        List of (x, y) pairs from the dataset

    Returns:
    --------
    tuple
        (stacked_xs, stacked_ys) as tensors
    """
    # zip(*batch) unpacks the list of (x,y) tuples
    # into (list of xs, list of ys)
    xs, ys = zip(*batch)
    # Stack into single tensors
    return torch.stack(xs), torch.stack(ys)

# Test collate
samples = [train_ds[i] for i in [0, 1, 2]]  # Get 3 samples
print("Individual samples:")
for i, (x, y) in enumerate(samples):
    print(f"  Sample {i}: x shape {x.shape}, y = {y.item()}")

xs, ys = collate(samples)
print(f"\nAfter collate:")
print(f"  xs shape: {xs.shape}")
print(f"  ys shape: {ys.shape}")

In [ ]:
# ============================================================================
# DATALOADER WITH SAMPLER SUPPORT
# ============================================================================

class DataLoader:
    """
    DataLoader that uses a BatchSampler for flexible iteration.

    Supports:
    - Custom batch sampling (sequential or shuffled)
    - Custom collate functions
    """
    def __init__(self, ds, batchs, collate_fn=collate):
        fc.store_attr()

    def __iter__(self):
        """Yield collated batches.""        # For each batch of indices from the batch sampler
        for b in self.batchs:
            # Get samples from dataset and collate them
            yield self.collate_fn(self.ds[i] for i in b)

# Create samplers and dataloaders
train_samp = BatchSampler(Sampler(train_ds, shuffle=True), bs)
valid_samp = BatchSampler(Sampler(valid_ds, shuffle=False), bs)

train_dl = DataLoader(train_ds, batchs=train_samp)
valid_dl = DataLoader(valid_ds, batchs=valid_samp)

print("DataLoaders with shuffling created!")

# Test
xb, yb = next(iter(train_dl))
print(f"\nTraining batch: xb shape {xb.shape}, yb shape {yb.shape}")

# Train
model, opt = get_model()
print("\nTraining with shuffled DataLoader...")
print("Format: loss, accuracy")
fit()

### PyTorch's Built-in DataLoader

PyTorch provides all this functionality (and much more) built-in! Let's use PyTorch's DataLoader with its samplers.

In [ ]:
# ============================================================================
# PYTORCH'S BUILT-IN DATALOADER
# ============================================================================
#|export

from torch.utils.data import DataLoader, SequentialSampler, RandomSampler, BatchSampler

# PyTorch provides these samplers:
# - SequentialSampler: Access samples in order (0, 1, 2, ...)
# - RandomSampler: Access samples in random order

# Create batch samplers using PyTorch's classes
train_samp = BatchSampler(RandomSampler(train_ds), bs, drop_last=False)
valid_samp = BatchSampler(SequentialSampler(valid_ds), bs, drop_last=False)

# Create DataLoaders
train_dl = DataLoader(train_ds, batch_sampler=train_samp, collate_fn=collate)
valid_dl = DataLoader(valid_ds, batch_sampler=valid_samp, collate_fn=collate)

# Train
model, opt = get_model()
print("Training with PyTorch DataLoader...")
print("Format: loss, accuracy")
fit()
print(f"\nFinal: loss={loss_func(model(xb), yb):.4f}, accuracy={accuracy(model(xb), yb):.2%}")

### The Simplest DataLoader Usage

PyTorch DataLoader can auto-generate samplers for us. This is the most common usage:

In [ ]:
# ============================================================================
# SIMPLEST DATALOADER USAGE
# ============================================================================

# PyTorch can generate samplers automatically!
# shuffle=True -> RandomSampler
# shuffle=False (default) -> SequentialSampler
# num_workers -> parallel data loading (very useful for slow data loading)

train_dl = DataLoader(train_ds, bs, shuffle=True, drop_last=True, num_workers=0)
valid_dl = DataLoader(valid_ds, bs, shuffle=False, num_workers=0)

# Note: num_workers=0 means load in main process
# Use num_workers>0 for parallel loading (faster but can cause issues on Windows)

print("Simple PyTorch DataLoader:")
print(f"  Training: shuffle=True, drop_last=True")
print(f"  Validation: shuffle=False")

# Train
model, opt = get_model()
print("\nTraining...")
print("Format: loss, accuracy")
fit()

print(f"\nFinal: loss={loss_func(model(xb), yb):.4f}, accuracy={accuracy(model(xb), yb):.2%}")

---

## Part 6: Validation - Checking for Overfitting

You **must** always have a **validation set** to check if your model is overfitting!

### What is Overfitting?

**Overfitting** happens when the model:
- Memorizes the training data
- But fails to generalize to new, unseen data

It's like a student who memorizes answers to practice problems but can't solve new problems.

```
+-----------------------------------------------------------------------------+
|                     TRAINING vs VALIDATION                                   |
+-----------------------------------------------------------------------------+
|                                                                             |
|   TRAINING SET (50,000 images):                                             |
|   - Used to update weights                                                  |
|   - Model sees these during training                                        |
|   - Loss goes down as model learns                                          |
|                                                                             |
|   VALIDATION SET (10,000 images):                                           |
|   - NOT used for training                                                   |
|   - Model never sees these during training                                  |
|   - Measures how well model generalizes                                     |
|                                                                             |
|   GOOD: Both training and validation metrics improve                        |
|   BAD:  Training improves but validation gets worse (overfitting!)          |
|                                                                             |
+-----------------------------------------------------------------------------+
```

### model.train() vs model.eval()

Some layers behave differently during training vs inference:
- **Dropout**: Randomly drops neurons during training, uses all during inference
- **BatchNorm**: Uses batch statistics during training, running stats during inference

Always call:
- `model.train()` before training
- `model.eval()` before validation/inference

For our simple model without Dropout/BatchNorm, this doesn't matter much, but it's a good habit!

In [ ]:
# ============================================================================
# THE COMPLETE FIT FUNCTION WITH VALIDATION
# ============================================================================
#|export

def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
    """
    Train a model and validate after each epoch.

    Parameters:
    -----------
    epochs : int
        Number of training epochs
    model : nn.Module
        The neural network model
    loss_func : callable
        Loss function (e.g., F.cross_entropy)
    opt : optimizer
        Optimizer (e.g., optim.SGD)
    train_dl : DataLoader
        DataLoader for training data
    valid_dl : DataLoader
        DataLoader for validation data

    Returns:
    --------
    tuple
        Final (validation_loss, validation_accuracy)
    """
    for epoch in range(epochs):
        # =====================================================================
        # TRAINING PHASE
        # =====================================================================
        model.train()  # Set model to training mode
                       # (Affects Dropout, BatchNorm, etc.)

        for xb, yb in train_dl:
            # Forward pass
            loss = loss_func(model(xb), yb)
            # Backward pass
            loss.backward()
            # Update weights
            opt.step()
            # Zero gradients
            opt.zero_grad()

        # =====================================================================
        # VALIDATION PHASE
        # =====================================================================
        model.eval()  # Set model to evaluation mode

        # torch.no_grad() disables gradient computation
        # - Faster (don't need to store computation graph)
        # - Uses less memory
        # - We don't need gradients during evaluation!
        with torch.no_grad():
            # Accumulate loss and accuracy across all validation batches
            tot_loss = 0.0
            tot_acc = 0.0
            count = 0

            for xb, yb in valid_dl:
                pred = model(xb)
                n = len(xb)
                count += n
                # .item() converts single-value tensor to Python number
                tot_loss += loss_func(pred, yb).item() * n
                tot_acc += accuracy(pred, yb).item() * n

        # Print epoch number, average loss, average accuracy
        avg_loss = tot_loss / count
        avg_acc = tot_acc / count
        print(f"Epoch {epoch}: loss={avg_loss:.4f}, accuracy={avg_acc:.2%}")

    return avg_loss, avg_acc

print("fit() function with validation defined!")
print("\nThis function:")
print("  1. Trains on training data each epoch")
print("  2. Evaluates on validation data after each epoch")
print("  3. Returns final validation metrics")

In [ ]:
# ============================================================================
# HELPER FUNCTION TO CREATE DATALOADERS
# ============================================================================
#|export

def get_dls(train_ds, valid_ds, bs, **kwargs):
    """
    Create training and validation DataLoaders.

    Parameters:
    -----------
    train_ds : Dataset
        Training Dataset
    valid_ds : Dataset
        Validation Dataset
    bs : int
        Batch size for training
    **kwargs : dict
        Additional arguments passed to DataLoader (e.g., num_workers)

    Returns:
    --------
    tuple
        (train_dl, valid_dl) - DataLoaders for training and validation

    Notes:
    - Training uses shuffle=True (randomize order each epoch)
    - Validation uses 2x batch size (can fit more since no gradients stored)
    - Validation doesn't shuffle (order doesn't matter for evaluation)
    """
    return (
        DataLoader(train_ds, batch_size=bs, shuffle=True, **kwargs),
        DataLoader(valid_ds, batch_size=bs*2, **kwargs)  # 2x batch size for validation!
    )

print("get_dls() helper function defined!")
print("\nNotes:")
print("  - Training: shuffle=True")
print("  - Validation: 2x batch size (no gradients = more memory available)")

### Putting It All Together

Now our complete training pipeline is just a few lines:

In [ ]:
# ============================================================================
# THE COMPLETE TRAINING PIPELINE
# ============================================================================
# Our whole process is now just 3 lines!

# 1. Create DataLoaders
train_dl, valid_dl = get_dls(train_ds, valid_ds, bs)

# 2. Create model and optimizer
model, opt = get_model()

# 3. Train with validation!
print("="*60)
print("TRAINING WITH VALIDATION")
print("="*60)
loss, acc = fit(5, model, loss_func, opt, train_dl, valid_dl)

print("="*60)
print(f"Final Validation Results:")
print(f"  Loss:     {loss:.4f}")
print(f"  Accuracy: {acc:.2%}")
print("="*60)

---

## Summary: What We Learned

We built a complete neural network training pipeline from scratch!

### The Complete Journey

```
+-----------------------------------------------------------------------------+
|                        MINIBATCH TRAINING SUMMARY                           |
+-----------------------------------------------------------------------------+
|                                                                             |
|   PART 1-2: SETUP & MODEL                                                   |
|   -----------------------                                                   |
|   - Loaded MNIST data (50k train, 10k valid images)                         |
|   - Built simple MLP: 784 -> 50 -> 10                                       |
|                                                                             |
|   PART 2: CROSS-ENTROPY LOSS                                                |
|   ---------------------------                                               |
|   - Softmax: Convert logits to probabilities                                |
|   - LogSumExp trick: Numerical stability                                    |
|   - NLL Loss: -log(p[correct_class])                                       |
|   - F.cross_entropy: Combined, efficient version                            |
|                                                                             |
|   PART 3: TRAINING LOOP                                                     |
|   ----------------------                                                    |
|   - Minibatches: Process data in small chunks                               |
|   - Forward -> Loss -> Backward -> Update -> Zero gradients                 |
|   - Repeat for all batches (epoch) x multiple epochs                        |
|                                                                             |
|   PART 4: OPTIMIZERS                                                        |
|   -------------------                                                       |
|   - model.parameters(): Auto-find all learnable params                      |
|   - optim.SGD: Stochastic Gradient Descent optimizer                        |
|   - opt.step() / opt.zero_grad(): Clean update code                         |
|                                                                             |
|   PART 5: DATASET & DATALOADER                                              |
|   -----------------------------                                             |
|   - Dataset: __getitem__ and __len__                                        |
|   - DataLoader: Batching, shuffling, parallelism                            |
|   - Sampler: Controls the order of samples                                  |
|                                                                             |
|   PART 6: VALIDATION                                                        |
|   ------------------                                                        |
|   - Separate validation set to check generalization                         |
|   - model.train() vs model.eval()                                           |
|   - torch.no_grad() during evaluation                                       |
|                                                                             |
+-----------------------------------------------------------------------------+
```

### Key Functions We Built

| Function | Purpose |
|----------|---------|
| `accuracy(out, yb)` | Compute classification accuracy |
| `report(loss, preds, yb)` | Print loss and accuracy |
| `fit(epochs, model, ...)` | Complete training loop with validation |
| `get_dls(train_ds, valid_ds, bs)` | Create training and validation DataLoaders |

### Key PyTorch Components

| Component | What It Does |
|-----------|--------------|
| `nn.Module` | Base class for all neural network modules |
| `nn.Linear` | Fully connected layer |
| `nn.Sequential` | Container for sequential layers |
| `F.cross_entropy` | Loss function for classification |
| `torch.optim.SGD` | Stochastic Gradient Descent optimizer |
| `DataLoader` | Batches and shuffles data |

### What's Next?

In the following notebooks, we'll learn:
- More advanced optimizers (Adam, AdamW)
- Learning rate schedulers
- Data augmentation
- Regularization techniques (Dropout, Weight Decay)
- More complex architectures (CNNs, ResNets)

**Congratulations! You now know how to train a neural network from scratch!**

---

## Export

The following cell exports the key functions to a Python module using nbdev.

In [ ]:
# ============================================================================
# EXPORT TO PYTHON MODULE
# ============================================================================
# nbdev.nbdev_export() extracts all cells marked with #|export
# and saves them to a .py file

import nbdev
nbdev.nbdev_export()

print("Exported!")